# Día 4: Introducción práctica a Streamlit

En esta notebook vamos a aprender a construir aplicaciones web sencillas con `Streamlit`.

El objetivo de esta sesión no es crear todavía la app final del curso, sino entender bien el flujo general de trabajo para que luego podamos reutilizarlo en otros proyectos.

Al terminar, deberías ser capaz de:

- crear un script `.py`;
- ejecutarlo con `streamlit run`;
- añadir widgets interactivos;
- organizar la interfaz;
- conservar información entre interacciones;
- evitar recálculos costosos con caché;
- cargar archivos e imágenes.

## Qué es Streamlit y cuándo usarlo

`Streamlit` es una librería de Python que permite convertir un script en una aplicación web interactiva con muy poco código.

Es útil cuando queremos:

- enseñar resultados de forma visual;
- crear demos rápidas;
- explorar datos;
- construir prototipos;
- envolver un modelo o algoritmo dentro de una interfaz sencilla.

No sustituye a un desarrollo web completo, pero para docencia, visualización y prototipado rápido es una herramienta excelente.

## Instalación

Si todavía no tienes las librerías necesarias, puedes instalarlas así:

```bash
pip install streamlit pandas numpy altair pillow matplotlib
```

## Flujo de trabajo

Una aplicación de `Streamlit` suele seguir este flujo:

1. escribes un archivo `.py`;
2. lo ejecutas con `streamlit run`;
3. se abre una app local en el navegador;
4. cuando guardas el archivo o cambias un widget, el script se vuelve a ejecutar.

Este último punto es la idea más importante de toda la sesión.

## Preparar una carpeta con ejemplos

En lugar de mezclar todos los ejemplos dentro de una sola app, vamos a crear varios scripts pequeños. Eso facilita mucho la enseñanza y también la depuración.

In [15]:
from pathlib import Path

examples_dir = Path("streamlit_examples")
examples_dir.mkdir(exist_ok=True)

examples_dir

WindowsPath('streamlit_examples')

## Cómo escribir archivos desde la notebook

En vez de guardar el código como cadenas de texto, vamos a usar `%%writefile` para escribir directamente cada ejemplo en un archivo `.py`.

Esto tiene dos ventajas:

- el código se edita como Python normal dentro de la celda;
- el resaltado de sintaxis es mucho más útil.

## Ejemplo 1: una app mínima

Empezamos con la app más simple posible: un título, un texto, una tabla y una gráfica.

In [16]:
%%writefile streamlit_examples/01_primera_app.py
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Primer ejemplo", layout="centered")

st.title("Mi primera app con Streamlit")
st.write("Esta aplicación está escrita solo con Python.")

df = pd.DataFrame(
    {
        "hora": [1, 2, 3, 4, 5],
        "temperatura": [21.2, 22.5, 23.1, 22.8, 21.9],
    }
)

st.metric("Temperatura media", f"{df['temperatura'].mean():.1f} C")
st.dataframe(df, use_container_width=True)
st.line_chart(df.set_index("hora"))

# Ejecuta con: streamlit run streamlit_examples/01_primera_app.py

Overwriting streamlit_examples/01_primera_app.py


Para ejecutarlo:

```bash
streamlit run streamlit_examples/01_primera_app.py
```

## Qué estamos usando aquí

- `st.set_page_config()` configura opciones generales;
- `st.title()` escribe un título principal;
- `st.write()` muestra texto y objetos;
- `st.metric()` destaca un valor resumido;
- `st.dataframe()` muestra tablas interactivas;
- `st.line_chart()` dibuja una gráfica rápida.

## El modelo mental correcto

Una app de `Streamlit` no funciona como una interfaz de escritorio tradicional. Aquí lo importante es pensar así:

- la app es un script normal de Python;
- `Streamlit` ejecuta ese script de arriba abajo;
- los widgets devuelven valores;
- al interactuar, el script se vuelve a ejecutar.

Si entiendes esto, casi todo lo demás resulta natural.

## Ejemplo 2: widgets básicos

Ahora vamos a añadir interacción. Los widgets son la base de cualquier app útil.

In [17]:
%%writefile streamlit_examples/02_widgets.py
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Widgets", layout="wide")

st.title("Widgets básicos")
st.write("Mueve los controles y observa cómo cambia la salida.")

num_points = st.slider("Número de puntos", min_value=20, max_value=200, value=60, step=20)
noise = st.slider("Ruido", min_value=0.0, max_value=1.0, value=0.15, step=0.05)
chart_type = st.selectbox("Tipo de gráfica", ["Líneas", "Área", "Barras"])
show_table = st.checkbox("Mostrar tabla", value=True)

rng = np.random.default_rng(7)
x = np.arange(num_points)
y = np.sin(x / 8) + rng.normal(0, noise, size=num_points)

df = pd.DataFrame({"x": x, "senal": y})

st.write(f"Has generado {num_points} puntos con un nivel de ruido de {noise:.2f}.")

if show_table:
    st.dataframe(df, use_container_width=True)

plot_df = df.set_index("x")

if chart_type == "Líneas":
    st.line_chart(plot_df)

if chart_type == "Área":
    st.area_chart(plot_df)

if chart_type == "Barras":
    st.bar_chart(plot_df)

# Ejecuta con: streamlit run streamlit_examples/02_widgets.py

Overwriting streamlit_examples/02_widgets.py


```bash
streamlit run streamlit_examples/02_widgets.py
```

## Widgets frecuentes

Los widgets más habituales al empezar son:

- `st.slider()`;
- `st.number_input()`;
- `st.text_input()`;
- `st.selectbox()`;
- `st.multiselect()`;
- `st.checkbox()`;
- `st.button()`.

## Ejemplo 3: organizar mejor la interfaz

Cuando la app crece, necesitamos estructura visual. Las herramientas más útiles al empezar son la barra lateral, las columnas y los bloques desplegables.

In [18]:
%%writefile streamlit_examples/03_layout.py
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Layout", layout="wide")

st.title("Diseño básico de una app")

st.sidebar.header("Controles")
amplitude = st.sidebar.slider("Amplitud", 0.5, 3.0, 1.2, 0.1)
frequency = st.sidebar.slider("Frecuencia", 0.5, 4.0, 1.0, 0.1)

x = np.linspace(0, 4 * np.pi, 300)
y = amplitude * np.sin(frequency * x)
df = pd.DataFrame({"x": x, "y": y}).set_index("x")

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Señal")
    st.line_chart(df)

with col2:
    st.subheader("Resumen")
    st.metric("Máximo", f"{y.max():.2f}")
    st.metric("Mínimo", f"{y.min():.2f}")
    st.metric("Media", f"{y.mean():.2f}")

with st.expander("Ver una muestra de los datos"):
    st.dataframe(df.reset_index().head(12), use_container_width=True)

# Ejecuta con: streamlit run streamlit_examples/03_layout.py

Overwriting streamlit_examples/03_layout.py


## Reglas prácticas de layout

- coloca en la barra lateral los controles secundarios;
- deja en el cuerpo principal lo más importante;
- usa columnas para comparar resultados;
- no abuses de demasiados widgets visibles a la vez;
- usa `st.expander()` para detalles opcionales.

## Ejemplo 4: entradas de texto y formularios

Cuando tenemos varios widgets relacionados, a veces no queremos que la app se reejecute visiblemente en cada cambio. Para eso existen los formularios.

In [19]:
%%writefile streamlit_examples/04_formularios.py
import streamlit as st

st.set_page_config(page_title="Formularios", layout="centered")

st.title("Widgets de entrada y formularios")

name = st.text_input("Nombre")
age = st.number_input("Edad", min_value=0, max_value=120, value=18)
role = st.selectbox("Perfil", ["Estudiante", "Docente", "Investigador"])

st.write("Vista inmediata")
st.write({"nombre": name, "edad": age, "perfil": role})

st.divider()
st.subheader("Formulario")

with st.form("datos_usuario"):
    city = st.text_input("Ciudad")
    likes_python = st.checkbox("Me gusta Python", value=True)
    submitted = st.form_submit_button("Enviar formulario")

if submitted:
    st.success("Formulario enviado")
    st.write({"ciudad": city, "python": likes_python})

# Ejecuta con: streamlit run streamlit_examples/04_formularios.py

Overwriting streamlit_examples/04_formularios.py


## Cuándo usar un formulario

Un formulario tiene sentido cuando:

- hay varias entradas relacionadas;
- quieres lanzar una acción solo al final;
- no quieres una reejecución visible por cada cambio.

## Ejemplo 5: `session_state`

Como el script se relanza completo, a veces necesitamos recordar cosas. Para eso usamos `st.session_state`.

In [20]:
%%writefile streamlit_examples/05_session_state.py
import streamlit as st

st.set_page_config(page_title="Estado", layout="centered")

st.title("Recordar información entre interacciones")

if "contador" not in st.session_state:
    st.session_state.contador = 0

if "historial" not in st.session_state:
    st.session_state.historial = []

col1, col2, col3 = st.columns(3)

with col1:
    if st.button("Sumar 1"):
        st.session_state.contador += 1
        st.session_state.historial.append(st.session_state.contador)

with col2:
    if st.button("Restar 1"):
        st.session_state.contador -= 1
        st.session_state.historial.append(st.session_state.contador)

with col3:
    if st.button("Reiniciar"):
        st.session_state.contador = 0
        st.session_state.historial = []

st.metric("Valor actual", st.session_state.contador)
st.write("Historial", st.session_state.historial)

# Ejecuta con: streamlit run streamlit_examples/05_session_state.py

Overwriting streamlit_examples/05_session_state.py


## Cuándo necesitas estado

`session_state` es útil para guardar:

- resultados intermedios;
- configuraciones del usuario;
- listas de acciones previas;
- imágenes subidas;
- predicciones calculadas previamente.

## Ejemplo 6: caché de datos

Si una función tarda en ejecutarse, no conviene repetirla cada vez que el usuario mueve un control.

In [21]:
%%writefile streamlit_examples/06_cache_data.py
import time
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Cache de datos", layout="centered")

st.title("Evitar recálculos con cache_data")

@st.cache_data
def create_dataframe(n_rows, seed):
    time.sleep(2)
    rng = np.random.default_rng(seed)
    return pd.DataFrame(
        {
            "x": np.arange(n_rows),
            "y": rng.normal(size=n_rows).cumsum(),
        }
    )

seed = st.number_input("Semilla", min_value=0, max_value=100, value=3, step=1)
n_rows = st.slider("Número de filas", min_value=100, max_value=1000, value=300, step=100)

st.write("La primera ejecución tarda. Si repites los mismos parámetros, será mucho más rápida.")

df = create_dataframe(n_rows, seed)
st.dataframe(df.head(10), use_container_width=True)
st.line_chart(df.set_index("x"))

# Ejecuta con: streamlit run streamlit_examples/06_cache_data.py

Overwriting streamlit_examples/06_cache_data.py


## `cache_data` frente a `cache_resource`

Regla práctica inicial:

- usa `@st.cache_data` para datos y resultados serializables;
- usa `@st.cache_resource` para recursos pesados compartidos, como un modelo cargado o una conexión.

## Ejemplo 7: gráficas con Altair y Matplotlib

`Streamlit` trae gráficas rápidas, pero también funciona bien con librerías externas.

In [22]:
%%writefile streamlit_examples/07_graficas.py
import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Graficas", layout="wide")

st.title("Integración con librerías de visualización")

x = np.linspace(0, 10, 200)
df = pd.DataFrame(
    {
        "x": x,
        "sin": np.sin(x),
        "cos": np.cos(x),
    }
)

tab1, tab2 = st.tabs(["Altair", "Matplotlib"])

with tab1:
    chart = alt.Chart(df).transform_fold(
        ["sin", "cos"],
        as_=["funcion", "valor"]
    ).mark_line().encode(
        x="x:Q",
        y="valor:Q",
        color="funcion:N"
    )
    st.altair_chart(chart, use_container_width=True)

with tab2:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df["x"], df["sin"], label="sin(x)")
    ax.plot(df["x"], df["cos"], label="cos(x)")
    ax.legend()
    ax.set_xlabel("x")
    ax.set_ylabel("valor")
    ax.set_title("Funciones trigonométricas")
    st.pyplot(fig)

# Ejecuta con: streamlit run streamlit_examples/07_graficas.py

Overwriting streamlit_examples/07_graficas.py


## Ejemplo 8: subir y visualizar archivos

Muchas apps reales empiezan por una subida de archivo.

In [23]:
%%writefile streamlit_examples/08_archivos.py
import numpy as np
import pandas as pd
from PIL import Image
import streamlit as st

st.set_page_config(page_title="Carga de archivos", layout="wide")

st.title("Subida de archivos")

tab1, tab2 = st.tabs(["CSV", "Imagen"])

with tab1:
    uploaded_csv = st.file_uploader("Sube un CSV", type=["csv"])
    if uploaded_csv is not None:
        df = pd.read_csv(uploaded_csv)
        st.dataframe(df, use_container_width=True)
        st.write("Dimensiones", df.shape)

with tab2:
    uploaded_image = st.file_uploader("Sube una imagen", type=["png", "jpg", "jpeg"])
    if uploaded_image is not None:
        image = Image.open(uploaded_image)
        st.image(image, caption="Imagen subida", use_container_width=True)
        st.write("Forma", np.array(image).shape)

# Ejecuta con: streamlit run streamlit_examples/08_archivos.py

Overwriting streamlit_examples/08_archivos.py


## Ejemplo 9: barra de progreso y mensajes de estado

Cuando una operación tarda, conviene decírselo al usuario.

In [24]:
%%writefile streamlit_examples/09_estado.py
import time
import streamlit as st

st.set_page_config(page_title="Estado de ejecucion", layout="centered")

st.title("Mensajes de estado")

if st.button("Simular proceso"):
    status = st.empty()
    bar = st.progress(0)

    for step in range(1, 11):
        status.write(f"Paso {step} de 10")
        bar.progress(step * 10)
        time.sleep(0.2)

    st.success("Proceso terminado")

# Ejecuta con: streamlit run streamlit_examples/09_estado.py

Overwriting streamlit_examples/09_estado.py


## Ejemplo 10: una mini app algo más realista

Aquí juntamos varias ideas en una sola aplicación: controles laterales, función de carga, caché, resumen y visualización.

In [25]:
%%writefile streamlit_examples/10_mini_app.py
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Mini app", layout="wide")

@st.cache_data
def generate_data(seed, n_rows):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame(
        {
            "grupo": rng.choice(["A", "B", "C"], size=n_rows),
            "valor_1": rng.normal(0, 1, size=n_rows),
            "valor_2": rng.normal(5, 2, size=n_rows),
        }
    )
    return df

st.sidebar.header("Parámetros")
seed = st.sidebar.slider("Semilla", 0, 50, 7)
n_rows = st.sidebar.slider("Número de filas", 50, 1000, 300, 50)
selected_group = st.sidebar.selectbox("Grupo", ["Todos", "A", "B", "C"])

df = generate_data(seed, n_rows)

if selected_group != "Todos":
    df = df[df["grupo"] == selected_group]

st.title("Ejemplo de mini app de análisis")

col1, col2, col3 = st.columns(3)
col1.metric("Filas", len(df))
col2.metric("Media valor_1", f"{df['valor_1'].mean():.2f}")
col3.metric("Media valor_2", f"{df['valor_2'].mean():.2f}")

tab1, tab2 = st.tabs(["Tabla", "Agregado por grupo"])

with tab1:
    st.dataframe(df, use_container_width=True)

with tab2:
    summary = df.groupby("grupo")[["valor_1", "valor_2"]].mean()
    st.bar_chart(summary)

# Ejecuta con: streamlit run streamlit_examples/10_mini_app.py

Overwriting streamlit_examples/10_mini_app.py


## Cómo lanzar cualquiera de los ejemplos

In [26]:
sorted(path.name for path in examples_dir.glob("*.py"))

['01_primera_app.py',
 '02_widgets.py',
 '03_layout.py',
 '04_formularios.py',
 '05_session_state.py',
 '06_cache_data.py',
 '07_graficas.py',
 '08_archivos.py',
 '09_estado.py',
 '10_mini_app.py']

```bash
streamlit run streamlit_examples/10_mini_app.py
```

## Consejos de organización

Cuando una app empieza a crecer, conviene separar:

- una zona de importaciones y configuración;
- funciones auxiliares;
- funciones de carga o preprocesado;
- el bloque de widgets;
- el bloque de visualización.

Un patrón sencillo sería:

In [27]:
import streamlit as st


def load_data():
    pass


def build_sidebar():
    pass


def main():
    pass


main()

## Buenas prácticas al empezar

- mantén scripts pequeños y legibles;
- usa nombres de variables claros;
- no mezcles carga, procesado y visualización sin orden;
- muestra mensajes claros cuando falten datos;
- usa caché solo donde aporte valor real;
- empieza por una versión simple y luego amplía.

## Errores frecuentes

Al empezar con `Streamlit`, es normal cometer estos fallos:

- olvidar que el script se reejecuta entero;
- recalcular continuamente algo pesado;
- perder variables por no usar `session_state` cuando hace falta;
- meter demasiados widgets sin estructura;
- intentar construir una app demasiado grande desde el principio.

## Ejercicio guiado 1

Crea una app llamada `11_reto_senal.py` con:

- un `slider` para amplitud;
- un `slider` para frecuencia;
- una gráfica de la señal generada;
- una métrica con el valor máximo.

## Ejercicio guiado 2

Crea una app llamada `12_reto_csv.py` con:

- subida de un CSV;
- visualización de la tabla;
- número de filas y columnas;
- selección de una columna numérica;
- una gráfica sencilla de esa columna.

## Ejercicio guiado 3

Crea una app llamada `13_reto_estado.py` con:

- un contador guardado en `session_state`;
- botones para sumar, restar y reiniciar;
- historial de valores;
- una casilla para mostrar u ocultar el historial.

## Despliegue básico

Durante el desarrollo usaremos normalmente:

```bash
streamlit run nombre_app.py
```

Más adelante una app de `Streamlit` puede compartirse en distintos entornos, pero para esta sesión lo importante es dominar bien el desarrollo local.

## Resumen final

La idea clave de `Streamlit` es muy simple:

1. escribes Python normal;
2. colocas elementos visuales con `st.*`;
3. recoges interacciones con widgets;
4. usas `session_state` para recordar cosas;
5. usas caché para no recalcular innecesariamente.

Si esto queda claro, luego resulta bastante natural envolver datasets, modelos o herramientas de análisis dentro de una aplicación interactiva.